# Cümle içindeki adresleri tanıma: Colab eğitimi

Bu defter, Türkiye OpenStreetMap adres kayıtlarını cümle kalıpları içine yerleştiren veriyle BERTurk modelini eğitir ve ayrı cümle testi üzerinde ölçer. Cümle kalıpları yapaydır; sonuçlar doğal müşteri mesajı başarısı olarak yorumlanmamalıdır.

## Başlamadan önce
1. Colab’da **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin (varsa).
2. Hücreleri sırayla çalıştırın. İlk hücrede bilgisayarınızdan `address-context-training-colab.zip` paketini seçin.
3. Eğitim ve test bitince son hücre model dosyalarını bilgisayarınıza indirir.


In [ ]:
import zipfile

from google.colab import files

uploaded = files.upload()
archive = next(name for name in uploaded if name.startswith("address-context-training-colab"))
project_dir = "/content/bitirme_maskeleme"
with zipfile.ZipFile(archive) as zf:
    zf.extractall(project_dir)
print("Paket açıldı:", archive)


In [ ]:
%cd /content/bitirme_maskeleme
!nvidia-smi
!pip -q install -r training/ner/requirements.txt


In [ ]:
!USE_TF=0 python training/ner/train.py \
  --data-dir data/ner/processed/real-address-context-v1 \
  --labels-path training/ner/labels.json \
  --output-dir /content/ner-address-context-model \
  --epochs 3 \
  --seed 42


## Cümle içindeki adresler için ayrı test

Aşağıdaki test adres kayıtlarını eğitimde kullanılmamış ayrı bir bölmeden alır. Cümle çevresi sabit, yapay kalıplarla oluşturulmuştur.


In [ ]:
!mkdir -p /content/address-context-eval
!cp data/ner/processed/openstreetmap-tr/sentence-context-v1/test.jsonl /content/address-context-eval/test.jsonl
!USE_TF=0 python training/ner/evaluate_model.py \
  --model-dir /content/ner-address-context-model/model \
  --data-dir /content/address-context-eval \
  --split test \
  --output-json /content/ner-address-context-model/context-test-metrics.json


## Modeli ve sonuçları indir

Bu hücre model dosyalarını, eğitim ölçümlerini ve cümle testi ölçümlerini tek ZIP dosyasına koyar.


## Modeli Drive'a yedekle

Colab oturumu kapanınca model silinmesin diye bu hücreyi çalıştır. Drive'da bitirme_maskeleme/model_runs/address_context_seed_42 klasörüne kaydeder.


In [ ]:
import shutil
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
source = Path('/content/ner-address-context-model')
destination = Path('/content/drive/MyDrive/bitirme_maskeleme/model_runs/address_context_seed_42')
run = 2
while destination.exists():
    destination = destination.with_name(f'address_context_seed_42_run_{run}')
    run += 1
shutil.copytree(source, destination)
print('Model ve ölçüm dosyaları Drive’a kaydedildi:', destination)


In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

out = Path("/content/ner-address-context-model")
archive = Path("/content/ner-address-context-model.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in ("metrics.json", "manifest.json", "context-test-metrics.json"):
        path = out / name
        if path.exists():
            zf.write(path, path.relative_to(out))
    for path in (out / "model").rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(out))
print("İndirilecek paket:", archive, "Boyut:", archive.stat().st_size, "byte")
files.download(str(archive))


## İlk yanlış alarm taraması: UCI yorumları

Değerlendirme ZIP'ini yükle. Kod 1.000 benzersiz yorumu sabit seed ile seçer ve yalnızca toplu sayıları yazar. Yorumlar etiketsiz olduğundan bunlar doğrulanmış yanlış pozitif değil, insanın inceleyeceği aday alarmlardır. Model dosyası önceki eğitim oturumundan yoksa önce eğitim hücrelerini yeniden çalıştır.


In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()  # address-alert-evaluation-colab.zip dosyasını seç
archive = next(name for name in uploaded if name.startswith('address-alert-evaluation-colab'))
eval_dir = Path('/content/address-alert-eval')
eval_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as zf:
    zf.extractall(eval_dir)
%cd /content/bitirme_maskeleme
!USE_TF=0 python /content/address-alert-eval/evaluate_address_alerts.py --model-dir /content/ner-address-context-model/model --uci-zip /content/address-alert-eval/turkish-user-review-dataset.zip --uci-count 1000 --seed 42 --show-review-candidates --output-json /content/uci-address-alert-report.json
files.download('/content/uci-address-alert-report.json')
